In [ ]:
import os
import pandas as pd
import yfinance as yf

# --- Download & save data in the same format expected by market.py (Date + Close) ---
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN"]

# yfinance output format has changed over time (often MultiIndex columns).
# This helper extracts the "Close" price robustly across formats.
def extract_close(download_df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(download_df.columns, pd.MultiIndex):
        # Common formats:
        # 1) columns = (Price, Ticker)   e.g., ("Close", "AAPL")
        # 2) columns = (Ticker, Price)   e.g., ("AAPL", "Close")
        lvl0 = set(map(str, download_df.columns.get_level_values(0)))
        lvl1 = set(map(str, download_df.columns.get_level_values(1)))
        if "Close" in lvl0:
            close_df = download_df["Close"]
        elif "Close" in lvl1:
            close_df = download_df.xs("Close", level=1, axis=1)
        else:
            raise ValueError("Couldn't find 'Close' in downloaded columns. Inspect download_df.columns.")
    else:
        # Single ticker case or older formats
        if "Close" in download_df.columns:
            close_df = download_df[["Close"]].copy()
        else:
            raise ValueError("Couldn't find 'Close' in downloaded columns. Inspect download_df.columns.")
    return close_df

data = yf.download(tickers, start="2010-01-01", auto_adjust=False, group_by="column", progress=False)
close = extract_close(data)

os.makedirs("data", exist_ok=True)

for ticker in tickers:
    if ticker not in close.columns:
        # In some edge cases yfinance may return a different naming, so we fail loudly.
        raise KeyError(f"Ticker '{ticker}' not found in Close data columns: {list(close.columns)}")
    single = close[[ticker]].reset_index()
    single.columns = ["Date", "Close"]

    # Match the legacy CSV date format found in the repository: dd-mm-YYYY
    single["Date"] = pd.to_datetime(single["Date"]).dt.strftime("%d-%m-%Y")

    single.to_csv(f"data/{ticker}.csv", index=False)


In [ ]:
import numpy as np
from market import market
from randomAgent import Agent
from momentumagent import MomentumAgent

In [ ]:
env = market(['AAPL', 'MSFT'])
agent = Agent(len(env.index_actions))

In [ ]:
def play_episode(agent, market):
    state = market.start()
    done = False

    while not done:
        action = agent.act(state)
        next_state, reward, done = market.new_day(action)
        state = next_state

    return market.get_episode_value()

In [ ]:
play_episode(agent,env)

np.float64(31963.81093835837)

In [ ]:
agent = MomentumAgent(len(env.index_actions), env.total_companies)
play_episode(agent,env)

np.float64(94591.77031707758)

## Desafio (Trabalho Final) — *Market RL Agent*

O objetivo é criares um **agente** capaz de interagir com o ambiente `market` (definido em `market.py`), tomando decisões de ação ao longo do tempo com base no estado observado.

> **Importante:** não se foquem demasiado em “ganhar” ao `RandomAgent`. O foco é **aprofundar um conceito da aula**, justificar decisões e mostrar pensamento crítico.  
> **Serás avaliado mais pela interpretação do que pelo melhor resultado.**

---

### Tarefa Principal

Escolhe **apenas UMA** das abordagens abaixo e resolve o problema.

#### Opção A — Monte Carlo (MC)
- Define claramente o que é um **episódio** no ambiente `market`.
- Escolhe MC **First-Visit** ou **Every-Visit**.
- Implementa:
  - política
  - avaliação/controlo (estimativa de \(V(s)\) ou \(Q(s,a)\))
  - mecanismo de exploração
- Discute: **variança**, necessidade de múltiplas amostras, impacto do tempo e escolha das recompensas.

#### Opção B — Temporal Difference (TD)
- Escolhe **TD(0)** / **SARSA** / **Q-Learning**.
- Implementa:
  - tabela de \(Q(s,a)\) (ou uma aproximação simples, se justificares)
  - exploração (ε-greedy)
  - learning rate (α) e discount (γ)
- Discute: **bootstrapping**, estabilidade, exploração vs exploração, convergência.

---

### Criatividade (Obrigatório)
Experimenta **pelo menos 2** escolhas “criativas”, por exemplo:
- **state abstraction** (ex.: discretização de retornos, bins, features simples)
- **reward shaping** (e justificar impactos)
- **multi-start** (várias corridas e comparar)
- **ε schedule** (fixo vs decaimento)
- **γ/α schedules** (constante vs adaptativo)
- mistura de métodos (ex.: MC para inicializar, TD para refinar)
- diferentes representações para o estado (mais/menos informação)

---

### Experiências Obrigatórias (para qualquer opção)
1. Corre o teu agente **várias vezes** (pelo menos **5** runs), com **seeds diferentes**.
2. Varia parâmetros relevantes (pelo menos **2 configurações** por agente).
3. Guarda e mostra o **histórico do valor/custo** ao longo do treino (learning curve).
4. Faz pelo menos uma comparação com um baseline simples (ex.: `RandomAgent` e/ou `MomentumAgent`).

**Gráficos obrigatórios:**
- Evolução do **retorno/valor do episódio** ao longo do treino (por episódio ou por iterações).
- Idealmente: inclui média móvel (ex.: janela 20) + dispersão (min/max ou std) entre runs.

**Outputs recomendados:**
- Gráficos: retorno por episódio (com várias seeds)
- Uma tabela pequena: configuração → melhor retorno → média/variância
- Comentário crítico: “o que funcionou / o que não funcionou / porquê”

---

### O que entregar

Um **Jupyter Notebook** contendo:
- Código (bem organizado e comentado)
- Experiências (várias runs, vários parâmetros)
- Interpretação crítica (texto)

Um **video** com uma breve apresentação do trabalho (~5 minutos) em que correm o trabalho e comentam as escolhas feitas  
(sugestão: OBS Studio para gravação do ecrã).

**Enviar o notebook e video da apresentação para (ivoacnogueira@gmail.com ou ivo.nogueira@isag.pt) até dia 15 de Março de 2026**.

---

### Regras

- Trabalho individual
- Código original
- Permitido: `numpy`, `pandas`, `matplotlib` (e bibliotecas standard do Python)
- Não usar bibliotecas RL/otimização prontas (ex.: `stable-baselines`, `gymnasium` trainers, `scipy.optimize`, etc.)
- Se usares uma aproximação de função (ex.: regressão simples), justifica claramente e mantém a implementação “manual” e pedagógica.